# Simple MultiAi Agent Architecture

### this researcher will be doing some websearch that output is given to the writer , writer has to write a summany
this the agent we are going to create

In [ ]:
import os 
from typing import TypedDict , Annotated , List , Literal
from langchain_core.messages import BaseMessage , HumanMessage , AIMessage , SystemMessage
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool 
from langchain_tavily import TavilySearch
from langgraph.graph import StateGraph , START , END , MessagesState
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import ToolNode

In [14]:
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [15]:
llm = init_chat_model(model="groq:openai/gpt-oss-120b")


In [ ]:
llm.invoke("what is 1+1")

AIMessage(content='1\u202f+\u202f1\u202f=\u202f2.', additional_kwargs={'reasoning_content': 'The user asks a simple math question: "what is 1+1". Answer: 2.'}, response_metadata={'token_usage': {'completion_tokens': 41, 'prompt_tokens': 77, 'total_tokens': 118, 'completion_time': 0.085797682, 'completion_tokens_details': {'reasoning_tokens': 22}, 'prompt_time': 0.004390641, 'prompt_tokens_details': None, 'queue_time': 0.01891192, 'total_time': 0.090188323}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_49bfac06f1', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019db62f-c22e-7112-9810-e31a13df6a16-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 77, 'output_tokens': 41, 'total_tokens': 118, 'output_token_details': {'reasoning': 22}})

: 

In [ ]:
# defining the state 
# the state will have the info about the whole conversation and which agent should go next
class AgentState(MessagesState):
    next_agent: str # which agent should go next

In [ ]:
# this researcher will be doing some websearch that output is given to the writer , writer has to write a summany

@tool
def search_web(query:str)->str:
    """Search the web for information."""
    # Using tavily search to search the web
    search =  TavilySearch(max_results=3)
    results = search.invoke(query)
    return str(results)


@tool
def write_summary(content: str):
    """Write a summary of the provided content"""
    # simple summary generation
    summary = f'Summary of findings:\n\n{content[:500]}...'
    return summary

In [ ]:
# define agent functions

def researcher_agent(state: AgentState):
    """Researcher agent that searches for information"""

    messages = state["messages"]

    # add system message for context
    system_msg = SystemMessage(content="You are a researcher assistant. Use the search_web tool to find information about the user request")

    # call LLM with tools
    researcher_llm = llm.bind_tools([search_web])
    response = researcher_llm.invoke([system_msg] + messages)

    return {
        'messages':[response] , 
        'next_agent':"writer"
    }

In [ ]:
def writer_agent(state: AgentState):
    """Writer agent that writes a summary of the research findings"""

    messages = state["messages"]

    # add system message for context
    system_msg = SystemMessage(content="You are a writer assistant. Use the write_summary tool to summarize the research findings provided by the researcher agent")

    # call LLM with tools
    writer_llm = llm.bind_tools([write_summary])
    response = writer_llm.invoke([system_msg] + messages)

    return {
        'messages':[response] , 
        'next_agent':"end"# end of the chain
    }